# Classification de gestes sportifs avec PyTorch
## Tutoriel 4 / 4

On prend les paramètres SMPL produits par le tutoriel 3 et on entraîne
un Conv1D pour distinguer des gestes.

```
smpl_outputs/
    forehand/
        video1/hmr4d_results.pt   ← body_pose (T, 69)
        video2/hmr4d_results.pt
    backhand/
        video1/hmr4d_results.pt
        ...
            ↓
        Conv1D PyTorch
            ↓
          Label
```

Le Conv1D opère directement sur la séquence temporelle de poses :
23 rotations articulaires × T frames.

In [ ]:
!pip install torch scikit-learn matplotlib --quiet

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

## 1. Charger les fichiers SMPL

Les fichiers viennent du tutoriel 3. Il faut les organiser par classe dans `smpl_outputs/` :
un sous-dossier par classe, puis un sous-dossier par vidéo contenant `hmr4d_results.pt`.

Si le tutoriel 3 a tourné sur un dossier `videos/forehand/` et `videos/backhand/`,
il suffit de lancer `save_smpl_params(..., output_dir="smpl_outputs/forehand")` pour chaque classe.

In [ ]:
# ── À MODIFIER ───────────────────────────────────────────────────────────────
SMPL_DIR = Path("smpl_outputs")   # dossier de sortie du tutoriel 3
SEQ_LEN  = 120                    # longueur fixe en frames (tronquer/padder)
# ─────────────────────────────────────────────────────────────────────────────

# Les classes = noms des sous-dossiers
classes   = sorted([d.name for d in SMPL_DIR.iterdir() if d.is_dir()])
LABEL_MAP = {cls: i for i, cls in enumerate(classes)}
print(f"Classes : {classes}")

sequences, labels = [], []
for cls in classes:
    files = list((SMPL_DIR / cls).rglob("hmr4d_results.pt"))
    print(f"  {cls} : {len(files)} fichiers")
    for f in files:
        data = torch.load(f, map_location="cpu")
        bp   = data["smpl_params_global"]["body_pose"]  # (T, 69)
        sequences.append(bp.numpy())
        labels.append(LABEL_MAP[cls])

print(f"\nTotal : {len(sequences)} séquences")

## 2. Dataset PyTorch

On tronque ou padde chaque séquence à `SEQ_LEN` frames.
Le Conv1D attend `(batch, channels, length)` donc on transpose :
`(T, 69)` → `(69, T)`.

In [ ]:
def pad_or_truncate(seq, length):
    T, D = seq.shape
    if T >= length:
        return seq[:length]
    return np.vstack([seq, np.zeros((length - T, D))])


class SMPLDataset(Dataset):
    def __init__(self, seqs, labels, seq_len):
        # (N, 69, T) — channels = joint angles, length = time
        self.X = torch.tensor(
            np.stack([pad_or_truncate(s, seq_len).T for s in seqs]),
            dtype=torch.float32
        )
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


y = np.array(labels)
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.25, stratify=y, random_state=42
)

train_ds = SMPLDataset([sequences[i] for i in idx_train], y[idx_train], SEQ_LEN)
test_ds  = SMPLDataset([sequences[i] for i in idx_test],  y[idx_test],  SEQ_LEN)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False)

N_CLASSES = len(classes)
print(f"Train : {len(train_ds)}   Test : {len(test_ds)}")
print(f"Shape entrée : {tuple(train_ds[0][0].shape)}  (channels=69, length={SEQ_LEN})")

## 3. Modèle Conv1D

Deux couches de convolution sur la dimension temporelle,
puis un global average pooling pour agréger en un vecteur fixe.

In [ ]:
class PoseConv1D(nn.Module):
    """
    Input  : (batch, 69, T)
    Output : (batch, N_classes)
    """
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(69,  128, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(128,  64, kernel_size=5, padding=2), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),   # (batch, 64, 1)
            nn.Flatten(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)


model   = PoseConv1D(N_CLASSES).to(DEVICE)
n_param = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {n_param:,}")

## 4. Entraînement

In [ ]:
opt     = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
history = {"train_acc": [], "test_acc": []}

for epoch in range(60):
    # — train —
    model.train()
    correct, total = 0, 0
    for xb, yb in train_loader:
        xb, yb  = xb.to(DEVICE), yb.to(DEVICE)
        logits  = model(xb)
        loss    = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        correct += (logits.detach().argmax(1) == yb).sum().item()
        total   += len(yb)

    # — eval —
    model.eval()
    correct_t, total_t = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            correct_t += (model(xb).argmax(1) == yb).sum().item()
            total_t   += len(yb)

    history["train_acc"].append(correct / total)
    history["test_acc"].append(correct_t / total_t)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/60  "
              f"train={history['train_acc'][-1]:.2%}  "
              f"test={history['test_acc'][-1]:.2%}")

## 5. Évaluation

In [ ]:
# Courbe d'apprentissage
plt.figure(figsize=(8, 4))
plt.plot(history["train_acc"], label="Train")
plt.plot(history["test_acc"],  label="Test", linestyle="--")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.ylim(0, 1.05); plt.legend(); plt.grid(alpha=0.3)
plt.title("Conv1D — accuracy")
plt.tight_layout(); plt.show()

# Matrice de confusion
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        y_pred.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())

y_true, y_pred = np.array(y_true), np.array(y_pred)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(N_CLASSES), classes, rotation=45)
plt.yticks(range(N_CLASSES), classes)
plt.xlabel("Prédit"); plt.ylabel("Réel")
plt.title("Matrice de confusion")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
plt.colorbar(); plt.tight_layout(); plt.show()

print(classification_report(y_true, y_pred, target_names=classes))

## 6. Comparaison et évaluation

In [ ]:
def get_predictions(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, sb, yb in loader:
            xb, sb = xb.to(DEVICE), sb.to(DEVICE)
            logits = model(xb, sb)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    return np.array(trues), np.array(preds)


CLASS_NAMES = list(CLASSES.keys())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, name in zip(axes, [mlp, lstm], ["MLP", "LSTM"]):
    y_true, y_pred = get_predictions(model, test_loader)
    cm = confusion_matrix(y_true, y_pred)

    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=45)
    ax.set_yticks(range(N_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
    ax.set_title(f"Matrice de confusion — {name}")
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14)
    plt.colorbar(im, ax=ax)

plt.tight_layout(); plt.show()

# Rapport détaillé
print("\n── MLP ──")
y_true, y_pred = get_predictions(mlp, test_loader)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

print("── LSTM ──")
y_true, y_pred = get_predictions(lstm, test_loader)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
def predict_gesture(video_path, model, scaler, use_lstm=False):
    """Prédit la classe d'un geste dans une vidéo."""
    seq  = video_to_pose_sequence(str(video_path), max_frames=SEQ_LEN)

    # Features MLP
    feat   = extract_features(seq)
    feat_n = scaler.transform(feat.reshape(1, -1))
    x_t    = torch.tensor(feat_n, dtype=torch.float32).to(DEVICE)

    # Séquence LSTM
    if len(seq) < SEQ_LEN:
        pad = np.zeros((SEQ_LEN - len(seq), seq.shape[1]))
        seq = np.vstack([seq, pad])
    else:
        seq = seq[:SEQ_LEN]
    s_t = torch.tensor(seq[None], dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(x_t, s_t)
        probs  = torch.softmax(logits, dim=-1)[0]
        pred   = probs.argmax().item()

    print(f"Vidéo : {Path(video_path).name}")
    for i, cls in enumerate(CLASS_NAMES):
        print(f"  {cls:12s} : {probs[i].item():.1%}")
    print(f"  → Prédiction : {CLASS_NAMES[pred].upper()}")
    return CLASS_NAMES[pred]


# Test sur les vidéos de test
print("Prédictions sur quelques exemples de test :\n")
for i in idx_test[:4]:
    # Récupérer le chemin depuis names_all
    # (chercher la vidéo dans le dataset)
    cls   = CLASS_NAMES[y[i]]
    fpath = class_files[cls][0]  # exemple
    predict_gesture(fpath, lstm, scaler, use_lstm=True)
    print()